## Setup


In [ ]:
import base64
import os
from pprint import pprint

from dotenv import load_dotenv

from agent_base.core.messages import Message
from agent_base.core.types import ContentBlockType, DocumentContent, ImageContent, Role, SourceType, TextContent, ToolResultContent
from agent_base.providers.litellm.formatters import LiteLLMMessageFormatter
from agent_base.providers.litellm.litellm_config import LiteLLMConfig
from agent_base.providers.litellm.provider import LiteLLMProvider
from agent_base.tools.tool_types import ToolSchema

load_dotenv('../../../.env')

PRIMARY_MODEL = os.getenv('LITELLM_TEST_MODEL_PRIMARY')
SECONDARY_MODEL = os.getenv('LITELLM_TEST_MODEL_SECONDARY')
REASONING_MODEL = os.getenv('LITELLM_TEST_MODEL_REASONING', PRIMARY_MODEL or '')

assert PRIMARY_MODEL, 'Set LITELLM_TEST_MODEL_PRIMARY in .env or the notebook environment.'

formatter = LiteLLMMessageFormatter()
litellm_provider = LiteLLMProvider(formatter=formatter)

weather_schema = ToolSchema(
    name='get_weather',
    description='Get current weather for a city. Always call this tool when asked about weather.',
    input_schema={
        'type': 'object',
        'properties': {'city': {'type': 'string'}},
        'required': ['city'],
    },
)


## Provider Round Trip Tests

Canonical Message -> format -> litellm client -> parse -> Canonical Response


### Basic - Primary Model


In [ ]:
canonical_message = Message.user('Reply with exactly: PONG')

res = await litellm_provider.generate(
    system_prompt='You are a test bot. Follow instructions exactly.',
    messages=[canonical_message],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=LiteLLMConfig(max_tokens=64),
    max_retries=1,
    base_delay=0.1,
)

pprint(res.to_dict())
assert res.role == Role.ASSISTANT
assert res.provider == 'litellm'
assert res.model
assert res.usage is not None
assert res.usage.input_tokens > 0
assert res.usage.output_tokens > 0
assert any(block.content_block_type == ContentBlockType.TEXT for block in res.content)


### Basic - Secondary Model (Optional)


In [ ]:
if not SECONDARY_MODEL:
    print('Skipping secondary-model round trip: LITELLM_TEST_MODEL_SECONDARY not set.')
else:
    res = await litellm_provider.generate(
        system_prompt='You are a test bot. Follow instructions exactly.',
        messages=[canonical_message],
        tool_schemas=[],
        model=SECONDARY_MODEL,
        llm_config=LiteLLMConfig(max_tokens=64),
        max_retries=1,
        base_delay=0.1,
    )
    pprint(res.to_dict())
    assert res.role == Role.ASSISTANT
    assert res.provider == 'litellm'
    assert res.model
    assert res.usage is not None
    assert any(block.content_block_type == ContentBlockType.TEXT for block in res.content)


### Client Tools


In [ ]:
tool_prompt = Message.user("What's the weather in Paris?")

tool_res = await litellm_provider.generate(
    system_prompt='Always use the get_weather tool for weather questions. Never answer directly.',
    messages=[tool_prompt],
    tool_schemas=[weather_schema],
    model=PRIMARY_MODEL,
    llm_config=LiteLLMConfig(max_tokens=128),
    max_retries=1,
    base_delay=0.1,
)

pprint(tool_res.to_dict())
assert tool_res.stop_reason == 'tool_use'
tool_block = next(block for block in tool_res.content if block.content_block_type == ContentBlockType.TOOL_USE)
assert tool_block.tool_name == 'get_weather'
assert tool_block.tool_id


### Tool Result Follow-up


In [ ]:
tool_result_message = Message(
    role=Role.USER,
    content=[
        ToolResultContent(
            tool_name=tool_block.tool_name,
            tool_id=tool_block.tool_id,
            tool_result='Sunny, 22C',
        ),
        TextContent(text='Summarize that briefly.'),
    ],
)

tool_followup = await litellm_provider.generate(
    system_prompt='Use tool outputs when they are provided.',
    messages=[tool_prompt, tool_res, tool_result_message],
    tool_schemas=[weather_schema],
    model=PRIMARY_MODEL,
    llm_config=LiteLLMConfig(max_tokens=128),
    max_retries=1,
    base_delay=0.1,
)

pprint(tool_followup.to_dict())
combined_text = ' '.join(block.text for block in tool_followup.content if block.content_block_type == ContentBlockType.TEXT).lower()
assert combined_text
assert 'sunny' in combined_text or '22' in combined_text or 'paris' in combined_text


### Image Input


In [ ]:
img_b64 = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAHggJ/PchI7wAAAABJRU5ErkJggg=='

image_res = await litellm_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        ImageContent(media_type='image/png', source_type=SourceType.BASE64, data=img_b64),
        TextContent(text='What color is this image?'),
    ])],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=LiteLLMConfig(max_tokens=128),
    max_retries=1,
    base_delay=0.1,
)

pprint(image_res.to_dict())
assert any(block.content_block_type == ContentBlockType.TEXT for block in image_res.content)


### Plain Text Document Input


In [ ]:
doc_text = 'The speed of light is approximately 299,792,458 meters per second.'
doc_b64 = base64.b64encode(doc_text.encode()).decode()

doc_res = await litellm_provider.generate(
    system_prompt=None,
    messages=[Message(role=Role.USER, content=[
        DocumentContent(media_type='text/plain', source_type=SourceType.BASE64, data=doc_b64),
        TextContent(text='What speed is mentioned in this document?'),
    ])],
    tool_schemas=[],
    model=PRIMARY_MODEL,
    llm_config=LiteLLMConfig(max_tokens=128),
    max_retries=1,
    base_delay=0.1,
)

pprint(doc_res.to_dict())
doc_text_out = ' '.join(block.text for block in doc_res.content if block.content_block_type == ContentBlockType.TEXT)
assert doc_text_out
assert '299' in doc_text_out or 'speed of light' in doc_text_out.lower()


### Reasoning / Thinking (Optional)


In [ ]:
if os.getenv('LITELLM_ENABLE_REASONING_TEST') != '1' or not REASONING_MODEL:
    print('Skipping reasoning test: enable with LITELLM_ENABLE_REASONING_TEST=1.')
else:
    reasoning_res = await litellm_provider.generate(
        system_prompt=None,
        messages=[Message.user('What is 25 * 37?')],
        tool_schemas=[],
        model=REASONING_MODEL,
        llm_config=LiteLLMConfig(
            max_tokens=256,
            thinking={'type': 'enabled', 'budget_tokens': 256},
        ),
        max_retries=1,
        base_delay=0.1,
    )
    pprint(reasoning_res.to_dict())
    assert any(block.content_block_type == ContentBlockType.TEXT for block in reasoning_res.content)
    thinking_blocks = [block for block in reasoning_res.content if block.content_block_type == ContentBlockType.THINKING]
    print(f'Thinking blocks returned: {len(thinking_blocks)}')
